# Qwen3-VL DPO v3：NEU-DET 钢材表面缺陷定位（SFT v3 之后接 DPO）

背景：SFT v3（NEU-DET，2 epochs，train_loss 0.834）已过验收线——测试集 180 张/195 项/420 框，
基线 Acc@0.5 16.43%、mean IoU 0.2109；微调 v3 Acc@0.5 22.14%（+5.71pp）、mean IoU 0.325、格式 100%。
第一次微调全面超过基线。DPO 是用户原定的下一阶段：SFT 跑完、评估过线后再接。

本版设计（与 SFT v3 同口径，便于归因）：
1. **偏好对**：1534 条训练对 + 192 条验证对，现场从 NEU-DET 标注生成（`gen_dpo_neudet.py` 同款逻辑，
   随机种子 42，可复现；不读旧 jsonl，旧的纺织品 jsonl 与本阶段无关）。
   每条为（图，类别）样本：prompt 用评估同款指令（TEMPLATES[0]）；chosen = 该类全部真实框；
   rejected = 全部框各平移 ±5%（每框至少动 1 像素，保证 rejected != chosen）。
   训练对用 train split，验证对用 val split——test split 是评估集，绝不进训练。
2. **起点是 SFT v3 适配器**（Kaggle 数据集 `neudet-adapter-v3`），不是基座：DPO 在 SFT 模型上继续优化，
   参考模型即 SFT 模型本身。产出为新适配器 `neudet-adapter-dpo`，评估本挂它即可复测（同口径）。
3. **与 SFT v3 完全相同的环境与视觉处理**：transformers 锁定 4.57.1（5.x 破坏 Qwen3-VL 视觉路径）；
   pixel_values 是无 batch 维的切块张量 (256, 1536)，绝不能取 [0]（v2 的教训：只剩首个补丁，
   视觉信号基本是空的）；image_grid_thw 带 num_images 维，[0] 才对。
4. **训练前先验偏好对**：条数、逐类分布、rejected != chosen、框合法性，任一项异常即停，不烧 GPU。

验收线（NEU-DET 测试集，同评估本口径）：格式合规率接近 100%，且 Acc@IoU>=0.5 与 mean IoU
高于 SFT v3（22.14% / 0.325）。只有复测确实有提升，才把数字写进简历。


In [ ]:
!pip install -q "transformers==4.57.1" "trl[peft,vlm]==1.18.2" "peft==0.22.0" "datasets==5.0.1" "accelerate==1.16.0" "bitsandbytes==0.50.2" pillow
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # DPO FIX 2026-11-24: single GPU only - DPOTrainer rejects nn.DataParallel (T4 x2 exposes 2 GPUs, Trainer then wraps in DataParallel and DPOTrainer raises ValueError)
# NOTE: trl MUST be 1.18.2, NOT 0.23.0. In 0.23.0 the VLM DPO path is broken for Qwen-VL:
# it does pixel_values[0] (a single patch - the same bug class as SFT v2) and never passes
# image_grid_thw, so the forward crashes. 1.18.2 handles Qwen-VL natively via
# DataCollatorForVisionPreference, and with a PEFT policy + ref_model=None it uses the
# SFT model itself as the reference (a frozen "ref" copy of the SFT adapter).
# NOTE: datasets MUST be 5.0.1, NOT 5.2.0 - datasets 5.2.0 requires huggingface-hub>=1.31,
# which conflicts with transformers 4.57.1 (hub<1.0). 5.0.1 was verified to resolve
# with this exact set on 2026-11-24.
import os
# ---- Dataset source: Kaggle input first, HF as fallback ----
# HF 429: anonymous downloads from Kaggle egress IPs are rate-limited
# (SFT V3-V6 all died here). The dataset is attached as Kaggle input.
# Probe (2026-11-24) found the real mount path is nested:
# /kaggle/input/datasets/gmy723/neu-det-corrected (extracted files,
# NOT a zip) - the flat /kaggle/input/neu-det-corrected guess was wrong.
DATA_ROOT = None
for _cand in ['/kaggle/input/datasets/gmy723/neu-det-corrected',
              '/kaggle/input/neu-det-corrected']:
    _z = os.path.join(_cand, 'neu-det-corrected.zip')
    if os.path.isfile(_z):
        import zipfile
        DATA_ROOT = '/kaggle/working/neudet'
        print('unzipping dataset from', _z)
        zipfile.ZipFile(_z).extractall(DATA_ROOT)
        break
    if os.path.isdir(os.path.join(_cand, 'images')):
        DATA_ROOT = _cand
        break
if DATA_ROOT is None:
    from huggingface_hub import snapshot_download
    DATA_ROOT = snapshot_download(repo_id='KeenForgeAI/NEU-DET-corrected', repo_type='dataset')
print('NEU-DET ->', DATA_ROOT)
import transformers
assert transformers.__version__ == '4.57.1', f"pinned transformers 4.57.1 required, got {transformers.__version__}"
print('transformers', transformers.__version__, '(pinned: latest 5.x breaks the Qwen3-VL vision path)')
import trl
assert trl.__version__ == '1.18.2', f"pinned trl 1.18.2 required, got {trl.__version__}"
print('trl', trl.__version__)
# DPO needs the val split too (SFT only used train; eval uses test).
assert os.path.isdir(os.path.join(DATA_ROOT, 'labels', 'val')), 'labels/val missing - DPO valid pairs need the val split'
assert os.path.isdir(os.path.join(DATA_ROOT, 'images', 'val')), 'images/val missing - DPO valid pairs need the val split'
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
import glob, json, random
from PIL import Image

# ---- Build DPO preference pairs in-notebook (same logic as gen_dpo_neudet.py) ----
# Deterministic (seed 42): train first, then valid, one RNG - reproduces the
# local files neudet_dpo_train.json (1534) / neudet_dpo_valid.json (192) exactly.
CLS_CN = {'crazing': '裂纹', 'inclusion': '夹杂物', 'patches': '斑块',
          'pitted_surface': '麻点', 'rolled-in_scale': '氧化皮', 'scratches': '划痕'}
PROMPT_TMPL = '请识别图片中的钢材表面缺陷（{cn}），并用边界框标出所有位置。'
SEED, PERTURB_RATIO = 42, 0.05
PAIR_DIR = '/kaggle/working/dpo_pairs'
os.makedirs(PAIR_DIR, exist_ok=True)

def fmt_answer(boxes):
    return '<|object_ref_start|>' + ''.join(
        f'<|object_ref|>{x1},{y1},{x2},{y2}<|/object_ref|>' for x1, y1, x2, y2 in boxes
    ) + '<|object_ref_end|>'

def perturb_box(box, w, h, rng):
    x1, y1, x2, y2 = box
    bw, bh = x2 - x1, y2 - y1
    dx = rng.uniform(-PERTURB_RATIO * bw, PERTURB_RATIO * bw)
    dy = rng.uniform(-PERTURB_RATIO * bh, PERTURB_RATIO * bh)
    ix, iy = int(round(dx)), int(round(dy))
    if ix == 0 and iy == 0:
        ix = rng.choice([-1, 1])
    nx1 = max(0, min(x1 + ix, w - 1)); ny1 = max(0, min(y1 + iy, h - 1))
    nx2 = max(0, min(x2 + ix, w - 1)); ny2 = max(0, min(y2 + iy, h - 1))
    if nx1 >= nx2:
        nx2 = min(nx1 + 1, w - 1)
        if nx1 >= nx2: nx1 = nx2 - 1
    if ny1 >= ny2:
        ny2 = min(ny1 + 1, h - 1)
        if ny1 >= ny2: ny1 = ny2 - 1
    assert 0 <= nx1 < nx2 < w and 0 <= ny1 < ny2 < h, (box, (nx1, ny1, nx2, ny2), w, h)
    return (nx1, ny1, nx2, ny2)

def build_split(split, rng):
    split_dir = {'train': 'train', 'valid': 'val'}[split]
    lbl_dir = os.path.join(DATA_ROOT, 'labels', split_dir)
    img_dir = os.path.join(DATA_ROOT, 'images', split_dir)
    classes = open(os.path.join(DATA_ROOT, 'classes.txt')).read().split()
    records, skipped = [], {'no_image': 0, 'no_box': 0, 'dup_box': 0, 'bad_image': 0}
    for lp in sorted(glob.glob(os.path.join(lbl_dir, '*.txt'))):
        name = os.path.splitext(os.path.basename(lp))[0]
        ip = os.path.join(img_dir, name + '.jpg')
        if not os.path.exists(ip):
            skipped['no_image'] += 1; continue
        try:
            w, h = Image.open(ip).size
        except Exception:
            skipped['bad_image'] += 1; continue
        by_cls = {}
        for line in open(lp, encoding='utf-8'):
            p = line.strip().split()
            if len(p) < 5: continue
            c = int(p[0]); cx, cy, bw, bh = map(float, p[1:5])
            x1, y1 = int(round((cx - bw / 2) * w)), int(round((cy - bh / 2) * h))
            x2, y2 = int(round((cx + bw / 2) * w)), int(round((cy + bh / 2) * h))
            by_cls.setdefault(c, []).append((x1, y1, x2, y2))
        if not by_cls:
            skipped['no_box'] += 1; continue
        for c, boxes in sorted(by_cls.items()):
            cls_name = classes[c]
            uniq = sorted(set(boxes))
            skipped['dup_box'] += len(boxes) - len(uniq)
            chosen_boxes = uniq
            rejected_boxes = [perturb_box(b, w, h, rng) for b in uniq]
            assert rejected_boxes != chosen_boxes, 'rejected == chosen for %s' % ip
            records.append({'prompt': PROMPT_TMPL.format(cn=CLS_CN[cls_name]),
                            'chosen': fmt_answer(chosen_boxes),
                            'rejected': fmt_answer(rejected_boxes),
                            'image': os.path.join('images', split_dir, name + '.jpg'),
                            'cls': cls_name, 'n_boxes': len(uniq)})
    return records, skipped

_rng = random.Random(SEED)
for _split in ('train', 'valid'):
    _recs, _sk = build_split(_split, _rng)
    _out = os.path.join(PAIR_DIR, f'neudet_dpo_{_split}.json')
    with open(_out, 'w', encoding='utf-8') as f:
        json.dump(_recs, f, ensure_ascii=False)
    print(f'{_split}: {len(_recs)} pairs | skipped: {_sk} | wrote {_out}')


In [ ]:
# ---- Verify preference pairs BEFORE training (fail fast, do not burn GPU) ----
import re
from collections import Counter
box_re = re.compile(r'<\|object_ref\|>(\d+),(\d+),(\d+),(\d+)<\|/object_ref\|>')
EXPECTED = {'train': (1534, {'crazing': 240, 'patches': 272, 'inclusion': 303,
                             'pitted_surface': 240, 'rolled-in_scale': 240, 'scratches': 239}),
            'valid': (192, {'crazing': 30, 'patches': 33, 'inclusion': 39,
                            'pitted_surface': 30, 'rolled-in_scale': 30, 'scratches': 30})}
pairs = {}
for _split in ('train', 'valid'):
    with open(os.path.join(PAIR_DIR, f'neudet_dpo_{_split}.json'), encoding='utf-8') as f:
        pairs[_split] = json.load(f)
    _n, _per = EXPECTED[_split]
    assert len(pairs[_split]) == _n, f'{_split}: got {len(pairs[_split])} pairs, expected {_n}'
    assert dict(Counter(r['cls'] for r in pairs[_split])) == _per, f'{_split}: per-class mismatch'
    assert all(r['rejected'] != r['chosen'] for r in pairs[_split]), f'{_split}: some rejected == chosen'
    for r in pairs[_split]:
        cb, rb = box_re.findall(r['chosen']), box_re.findall(r['rejected'])
        assert len(cb) == len(rb) == r['n_boxes'], 'box count mismatch'
        for x1, y1, x2, y2 in rb:
            assert int(x1) < int(x2) and int(y1) < int(y2), 'invalid rejected box'
    print(f'{_split}: {len(pairs[_split])} pairs verified | per-class: {_per}')
# spot-check: the images referenced by the first 3 train pairs open fine
for r in pairs['train'][:3]:
    _im = Image.open(os.path.join(DATA_ROOT, r['image'])).convert('RGB')
    print('image OK:', r['image'], _im.size, '| cls:', r['cls'], '| boxes:', r['n_boxes'])
print('sample prompt:', pairs['train'][0]['prompt'])
print('sample chosen:', pairs['train'][0]['chosen'][:160])
print('sample rejected:', pairs['train'][0]['rejected'][:160])
print('PAIR CHECKS PASSED')


In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

BASE = 'Qwen/Qwen3-VL-2B-Instruct'
# ---- SFT v3 adapter: attach the neudet-adapter-v3 Kaggle dataset as input ----
ADAPTER_DIR = None
for _d, _dirs, _files in os.walk('/kaggle/input'):
    if 'adapter_config.json' in _files:
        ADAPTER_DIR = _d
        break
print('SFT adapter dir:', ADAPTER_DIR)
assert ADAPTER_DIR, 'SFT adapter not found - attach the neudet-adapter-v3 dataset as input'
processor = AutoProcessor.from_pretrained(BASE)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
base = Qwen3VLForConditionalGeneration.from_pretrained(
    BASE, quantization_config=bnb, device_map="cuda:0", dtype=torch.bfloat16)
base.config.use_cache = False
# Policy starts FROM the SFT adapter (same adapter keeps training, "default").
# TRL copies it into a frozen "ref" adapter at trainer init, so the reference
# model IS the SFT model - the correct DPO setup. No merge step needed.
model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=True)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
# DPO FIX 2026-11-24 (V3): PEFT 0.22.0 quirk, verified against its source - from_pretrained
# constructs the adapter (frozen, requires_grad_(False) in _inject_adapter) BEFORE
# load_adapter runs, so load_adapter's `if is_trainable: set_requires_grad()` branch is
# skipped (adapter already in peft_config) and is_trainable=True NEVER unfreezes anything.
# V2 proved it: trainable params stayed 0 with is_trainable=True. Unfreeze explicitly:
for _n, _p in model.named_parameters():
    _p.requires_grad = ('lora_' in _n)
model.train()
model.print_trainable_parameters()
_n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert _n_train > 0, 'ADAPTER FROZEN: trainable params = 0 (is_trainable=True required)'
print('trainable params:', _n_train)
print('policy = base + SFT adapter; reference = frozen SFT copy (handled by TRL)')


In [ ]:
from datasets import Dataset as HFDataset

# ---- Conversational format for TRL 1.18.2: prompt/chosen/rejected as message lists,
# images as PIL objects (NOT paths). TRL inserts the image block into the user
# message and applies the chat template itself; no custom collator is needed.
def to_conversational(records):
    rows = []
    for r in records:
        img = Image.open(os.path.join(DATA_ROOT, r['image'])).convert('RGB')
        rows.append({
            'prompt': [{'role': 'user', 'content': r['prompt']}],
            'chosen': [{'role': 'assistant', 'content': r['chosen']}],
            'rejected': [{'role': 'assistant', 'content': r['rejected']}],
            'images': [img],
        })
    return rows

train_ds = HFDataset.from_list(to_conversational(pairs['train']))
valid_ds = HFDataset.from_list(to_conversational(pairs['valid']))
print('train:', len(train_ds), '| valid:', len(valid_ds))
print('row 0 prompt:', train_ds[0]['prompt'])
print('row 0 images:', len(train_ds[0]['images']), type(train_ds[0]['images'][0]))
print('row 0 chosen:', train_ds[0]['chosen'])


In [ ]:
from trl import DPOConfig, DPOTrainer
from transformers import TrainerCallback

OUT_ADAPTER = '/kaggle/working/neudet-adapter-dpo'

class SmokeCheck(TrainerCallback):
    # The first real log (step 5) must show a finite loss and DPO rewards;
    # otherwise stop immediately - 5 steps burned, not a full epoch.
    _checked = False
    def on_log(self, args, state, control, logs=None, **kw):
        if self._checked or not logs:
            return
        self._checked = True
        loss = logs.get('loss')
        assert loss is not None and loss == loss, 'SMOKE FAIL: loss missing or NaN in first log'
        assert any(k.startswith('rewards/') for k in logs), 'SMOKE FAIL: no rewards/* logged'
        print(f"SMOKE OK at step {state.global_step}: loss={loss:.4f} "
              f"rewards/accuracies={logs.get('rewards/accuracies')}")

args = DPOConfig(
    output_dir=OUT_ADAPTER,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=5e-6,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    beta=0.1,
    loss_type='sigmoid',
    max_length=1024,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    logging_steps=5,
    save_strategy='no',
    eval_strategy='epoch',
    report_to=[],
    dataloader_num_workers=0,
    precompute_ref_log_probs=False,
    sync_ref_model=False,
)
trainer = DPOTrainer(
    model=model, ref_model=None, args=args,
    train_dataset=train_ds, eval_dataset=valid_ds,
    processing_class=processor, callbacks=[SmokeCheck()],
)
# ---- Vision-input self-check: fail fast, before training ----
_batch = trainer.data_collator([train_ds[0]])
print('collated keys:', sorted(_batch.keys()))
print('pixel_values shape:', tuple(_batch['pixel_values'].shape))
print('image_grid_thw:', _batch['image_grid_thw'])
assert _batch['pixel_values'].shape[1] == 256, 'PIXEL BUG: patch count != 256'
assert 'image_grid_thw' in _batch, 'MISSING image_grid_thw - forward will crash'
print('vision inputs OK: full patches + grid reach the model')
trainer.train()
print('training done')
trainer.save_model(OUT_ADAPTER)
hist = [{'step': h.get('step'), 'loss': h.get('loss'),
         'acc': h.get('rewards/accuracies')} for h in trainer.state.log_history if 'loss' in h]
print('first loss:', hist[0], '| last loss:', hist[-1])
with open('/kaggle/working/dpo_train_log.json', 'w', encoding='utf-8') as f:
    json.dump(hist, f, ensure_ascii=False, indent=2)
import glob as _glob
print('adapter files:', sorted(_glob.glob(os.path.join(OUT_ADAPTER, '*'))))
print('DPO DONE -> adapter at', OUT_ADAPTER)
# NOTE: the saved adapter IS the SFT adapter continued (same "default" adapter),
# so it already contains the SFT v3 weights. The eval notebook only needs the
# input dataset swapped to neudet-adapter-dpo - no code change.


## 下一步

1. 在 Kaggle 上把本笔记本的输出（`neudet-adapter-dpo`）作为输入数据挂到评估本
   `qwen3vl-eval-kaggle-v3-neudet.ipynb`（把适配器输入换成 DPO 的那份），Run all。
2. 对照同本内测出的三组数字：基线、SFT v3（Acc@0.5 22.14%、mean IoU 0.325）、DPO——
   看格式合规率是否仍接近 100%、Acc@IoU>=0.5 与 mean IoU 是否高于 SFT v3、是否还有重复框。
3. 只有复测确实有提升，才把数字写进简历；写法须注明是 NEU-DET 公开基准上的微调验证。
   DPO 阶段本身不直接产生简历数字，数字以评估本的复测为准。
